In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# =============================================================================
# CONFIGURACIÓN 
# =============================================================================

MZ = 91.1876          # masa del bosón Z en GeV (PDG)
CHUNKSIZE = 200_000   # filas por bloque al leer los CSV

ETIQUETA = "estricto"

# ---------- Rutas ----------
CARPETA = "D:/proyecto_cern/"
NOMBRE_CARPETA = "cutflow 70-180"          # carpeta de salida (dentro de CARPETA)
SALIDA = CARPETA + NOMBRE_CARPETA + "/"

# Los 4 CSV de datos reales (se procesan y se unen como un solo conjunto)
DATOS_REALES = [
    CARPETA + "Run2012B_DoubleElectron_Data_Aplanada.csv",
    CARPETA + "Run2012B_DoubleMuParked_Data_Aplanada.csv",
    CARPETA + "Run2012C_DoubleElectron_Data_Aplanada.csv",
    CARPETA + "Run2012C_DoubleMuParked_Data_Aplanada.csv",
]

# CSV de 4 leptones de las simulaciones 
RUTAS = {
    "SMHiggsToZZTo4L": CARPETA + "SMHiggs_4lep.csv",
    "ZZTo4mu":         CARPETA + "ZZTo4mu_4lep.csv",
    "ZZTo4e":          CARPETA + "ZZTo4e_4lep.csv",
    "ZZTo2e2mu":       CARPETA + "ZZTo2e2mu_4lep.csv",
}
MOSTRAR = False   

# ---------- Rango de m4l en tablas y gráficas ----------
VENTANA = (70.0, 180.0)   # (min, max) en GeV: solo se cuentan/grafican eventos con m4l
                          # dentro de este rango.
ANCHO_BIN = 5.0           
XMAX = None               # solo con VENTANA = None: límite del eje X
                          # (None = hasta el m4l más grande; o un número, ej. 500)

# ---------- Luminosidad y muestras simulados ----------
LUMI = 11580.0   # luminosidad integrada en pb^-1 (= 11.58 fb^-1)

# xsec : sección eficaz en pb
# nevt : eventos GENERADOS en la simulación (no las filas del CSV)
# k    : K-factor (corrección a órdenes superiores; 1.386 para el fondo ZZ)
# tipo : "senal" (Higgs) o "fondo" (ZZ); define cómo se dibuja en las gráficas
MUESTRAS = {
    "SMHiggsToZZTo4L": {"xsec": 0.0065, "nevt": 299973.0,  "k": 1.0,   "tipo": "senal"},
    "ZZTo4mu":         {"xsec": 0.077,  "nevt": 1499064.0, "k": 1.386, "tipo": "fondo"},
    "ZZTo4e":          {"xsec": 0.077,  "nevt": 1499093.0, "k": 1.386, "tipo": "fondo"},
    "ZZTo2e2mu":       {"xsec": 0.18,   "nevt": 1497445.0, "k": 1.386, "tipo": "fondo"},
}

# Nombres de las columnas en la tabla cutflow
NOMBRES_COLUMNA = {
    "SMHiggsToZZTo4L": "Higgs",
    "ZZTo4mu": "ZZ->4mu",
    "ZZTo4e": "ZZ->4e",
    "ZZTo2e2mu": "ZZ->2e2mu",
}

# =============================================================================
# VALORES DE LOS CORTES
# =============================================================================
# Los cortes de "por leptón" se exigen a LOS 4 leptones del evento.
CUTS = {
    # --- Cinemática mínima de cada leptón ---
    "pt_e_min": 7,        # electrones: pT > 7 GeV
    "pt_mu_min": 5,       # muones:     pT > 5 GeV
    "eta_e_max": 2.5,     # electrones: |eta| < 2.5 (cobertura del detector)
    "eta_mu_max": 2.4,    # muones:     |eta| < 2.4 (cobertura de las cámaras de muones)

    # --- Compatibilidad con el vértice primario (descarta leptones "no prompt") ---
    "dxy_max": 0.5,       # |dxy| < 0.5 cm
    "dz_max": 1.0,        # |dz|  < 1 cm
    "sip_max": 4.0,       # SIP3D < 4
    "iso_max": 0.4,       # aislamiento relativo < 0.4

    # --- Leptones más duros del evento ---
    "pt1_min": 20,        # el leptón de mayor pT debe tener pT > 20 GeV
    "pt2_min": 10,        # el segundo de mayor pT debe tener pT > 10 GeV

    # --- Masas de los pares (bosones Z) ---
    # Z1 = el par más cercano a la masa del Z; Z2 = el otro par.
    "mz1_min": 40, "mz1_max": 120,
    "mz2_min": 12, "mz2_max": 120,
    "mll_min": 4,         # cualquier par de carga opuesta debe tener masa > 4 GeV
                          # (elimina resonancias de quarks pesados de baja masa)
    "m4l_min": 70,        # masa de los 4 leptones > 70 GeV
}

BINS = None   # bordes de los bins; se define en el bloque principal (construir_bins)

# Etapas (filtros) en orden. Cada una es ACUMULATIVA (incluye las anteriores).
ETAPAS = [
    "Eventos de entrada",
    "4 leptones validos (sin -999/NaN)",
    "pT de los leptones",
    "|eta| de los leptones",
    "|dxy| y |dz|",
    "SIP3D",
    "Aislamiento relativo",
    "Carga total = 0",
    "pT1 y pT2 minimos",
    "Existe emparejamiento Z+Z (SF, OS)",   # SF = same flavour, OS = opposite sign
    "mZ1 en ventana",
    "mZ2 en ventana",
    "mll minimo (todos los pares OS)",
    "m4l minimo",
]

# Columnas que existen para cada leptón (sufijo 1..4 en el CSV)
COLS_LEP = ["PID", "Q", "pt", "eta", "phi", "mass", "dxy", "dxyErr", "dz", "dzErr",
            "pfRelIso03_all", "pfRelIso04_all"]
# Conjunto de columnas que se leen del CSV (las demás se ignoran para ahorrar memoria)
NECESARIAS = {f"{c}{i}" for c in COLS_LEP for i in range(1, 5)} | {"run", "luminosityBlock", "event"}
# Columnas que no pueden faltar para considerar "válido" un leptón (etapa 1)
COLS_VALIDAS = [f"{v}{i}" for v in ["PID", "Q", "pt", "eta", "phi", "mass"] for i in range(1, 5)]


# =============================================================================
# FÍSICA Y SELECCIÓN
# =============================================================================

def p4(df, i):
    """
    Cuadrivector (E, px, py, pz) del leptón i para todos los eventos del bloque.

    Se pasa de las variables (pt, eta, phi, masa) a componentes cartesianas:
        px = pT cos(phi),  py = pT sin(phi),  pz = pT sinh(eta)
        E  = sqrt(px^2 + py^2 + pz^2 + m^2)
    Devuelve un arreglo de forma (n_eventos, 4).
    """
    pt = df[f"pt{i}"].to_numpy()
    eta = df[f"eta{i}"].to_numpy()
    phi = df[f"phi{i}"].to_numpy()
    m = df[f"mass{i}"].to_numpy()
    px, py, pz = pt * np.cos(phi), pt * np.sin(phi), pt * np.sinh(eta)
    E = np.sqrt(px**2 + py**2 + pz**2 + m**2)
    return np.stack([E, px, py, pz], axis=1)


def inv_mass(*vecs):
    """
    Masa invariante de un sistema de partículas: suma los cuadrivectores y calcula
        m^2 = E^2 - px^2 - py^2 - pz^2
    (el clip a 0 evita raíces de números negativos pequeños por redondeo).
    """
    s = sum(vecs)
    m2 = s[:, 0]**2 - s[:, 1]**2 - s[:, 2]**2 - s[:, 3]**2
    return np.sqrt(np.clip(m2, 0, None))


def sip3d(dxy, dxyErr, dz, dzErr):
    """
    Significancia del parámetro de impacto en 3D (SIP3D) = IP / sigma_IP.

    Es una APROXIMACION: se toma IP = sqrt(dxy^2 + dz^2) y su incertidumbre se
    obtiene por propagación de errores SIN correlaciones entre dxy y dz (el CSV
    no trae la matriz de covarianza completa). Un SIP3D grande indica que el
    leptón no viene del vértice primario (probablemente de un decaimiento de
    quarks pesados), por lo que se descarta.
    """
    dxy = np.asarray(dxy, float)
    dxyErr = np.asarray(dxyErr, float)
    dz = np.asarray(dz, float)
    dzErr = np.asarray(dzErr, float)
    ip = np.sqrt(dxy**2 + dz**2)
    sig = np.sqrt((dxy**2 * dxyErr**2 + dz**2 * dzErr**2) / np.clip(ip**2, 1e-12, None))
    return ip / np.clip(sig, 1e-12, None)


def evaluar(df, C, con_claves=False):
    """
    Aplica TODOS los cortes, de forma acumulativa, a un bloque (chunk) del CSV.

    Parámetros
    ----------
    df         : bloque del CSV (DataFrame).
    C          : diccionario con los valores de los cortes (CUTS).
    con_claves : True para datos reales; guarda una clave única por evento
                 (hash de run, luminosityBlock, event) para quitar duplicados
                 entre datasets más adelante.

    Devuelve
    --------
    {etapa: (m4l de los sobrevivientes, claves de evento o None, n sobrevivientes)}

    Nota: si VENTANA no es None, en cada etapa solo se guardan/cuentan los
    eventos cuyo m4l cae dentro de la ventana. La etapa 0 ("Eventos de entrada")
    cuenta todas las filas del bloque, sin ventana.
    """
    res = {}
    claves = None
    if con_claves:
        claves = pd.util.hash_pandas_object(
            df[["run", "luminosityBlock", "event"]], index=False).to_numpy()
    res[ETAPAS[0]] = (None, claves, len(df))

    # ---------- Etapa 1: 4 leptones válidos ----------
    # El valor -999 es relleno ("no hay dato"); se convierte a NaN y se descartan
    # los eventos a los que les falte cualquier dato básico de algún leptón.
    df = df.replace(-999, np.nan)
    valido = df[COLS_VALIDAS].notna().all(axis=1).to_numpy()
    df = df[valido]
    if claves is not None:
        claves = claves[valido]
    n = len(df)

    # m4l (masa invariante de los 4 leptones) de todos los eventos válidos.
    # Se calcula aquí una sola vez para poder graficar desde la primera etapa.
    v = [p4(df, i) for i in range(1, 5)]            # cuadrivectores de los 4 leptones
    m4l = inv_mass(*v)
    pid = [df[f"PID{i}"].abs().to_numpy() for i in range(1, 5)]   # |PID|: 11 = e, 13 = mu
    q = [df[f"Q{i}"].to_numpy() for i in range(1, 5)]             # cargas

    # 'vivo' marca los eventos que han pasado TODOS los cortes hasta el momento.
    vivo = np.ones(n, bool)
    if VENTANA is None:
        en_ventana = np.ones(n, bool)
    else:
        en_ventana = (m4l >= VENTANA[0]) & (m4l <= VENTANA[1])

    def cerrar(nombre, mascara):
        """
        Cierra una etapa: aplica su máscara sobre 'vivo' (acumulativo) y guarda
        m4l, claves y conteo de los eventos vivos que además están en la ventana.
        """
        nonlocal vivo
        vivo = vivo & mascara
        sel = vivo & en_ventana
        res[nombre] = (m4l[sel].astype(np.float32),
                       None if claves is None else claves[sel],
                       int(sel.sum()))

    cerrar(ETAPAS[1], np.ones(n, bool))   # etapa 1: ya filtrada arriba, no corta más

    # ---------- Etapas 2-6: cortes por leptón ----------
    # Cada corte exige que LOS 4 leptones lo cumplan (por eso el &= dentro del bucle).
    pt_ok = np.ones(n, bool)
    eta_ok = np.ones(n, bool)
    ip_ok = np.ones(n, bool)
    sip_ok = np.ones(n, bool)
    iso_ok = np.ones(n, bool)
    for i in range(1, 5):
        es_e = pid[i - 1] == 11      # ¿el leptón i es electrón?
        es_mu = pid[i - 1] == 13     # ¿el leptón i es muón?
        pt = df[f"pt{i}"].to_numpy()
        eta = np.abs(df[f"eta{i}"].to_numpy())
        dxy = df[f"dxy{i}"].to_numpy(float)
        dz = df[f"dz{i}"].to_numpy(float)

        # pT mínimo (distinto para electrones y muones)
        pt_ok &= ((es_e & (pt > C["pt_e_min"])) | (es_mu & (pt > C["pt_mu_min"])))
        # |eta| máximo (distinto para electrones y muones)
        eta_ok &= ((es_e & (eta < C["eta_e_max"])) | (es_mu & (eta < C["eta_mu_max"])))
        # parámetros de impacto transversal y longitudinal respecto al vértice
        ip_ok &= (np.abs(dxy) < C["dxy_max"]) & (np.abs(dz) < C["dz_max"])

        # SIP3D (aproximado, ver sip3d). Si no se puede calcular (NaN), el leptón
        # NO se descarta por este corte.
        sip = sip3d(dxy, df[f"dxyErr{i}"], dz, df[f"dzErr{i}"])
        sip_ok &= (sip < C["sip_max"]) | np.isnan(sip)

        # Aislamiento relativo: los muones usan el cono de radio 0.4 (pfRelIso04);
        # los electrones no tienen esa columna (NaN), así que se usa el cono 0.3
        # (pfRelIso03). fillna rellena solo donde falta la de 0.4.
        iso = df[f"pfRelIso04_all{i}"].fillna(df[f"pfRelIso03_all{i}"]).to_numpy()
        iso_ok &= iso < C["iso_max"]

    cerrar(ETAPAS[2], pt_ok)
    cerrar(ETAPAS[3], eta_ok)
    cerrar(ETAPAS[4], ip_ok)
    cerrar(ETAPAS[5], sip_ok)
    cerrar(ETAPAS[6], iso_ok)

    # ---------- Etapa 7: carga total cero ----------
    # Los 4 leptones vienen de dos Z neutros, así que las cargas deben sumar 0.
    cerrar(ETAPAS[7], (q[0] + q[1] + q[2] + q[3]) == 0)

    # ---------- Etapa 8: pT de los dos leptones más duros ----------
    # Se ordenan los 4 pT de mayor a menor y se exige pT1 > 20 y pT2 > 10 GeV
    # (para asegurar que el evento pudo ser disparado por el trigger).
    pts = np.sort(np.stack([df[f"pt{i}"].to_numpy() for i in range(1, 5)], axis=1), axis=1)[:, ::-1]
    cerrar(ETAPAS[8], (pts[:, 0] > C["pt1_min"]) & (pts[:, 1] > C["pt2_min"]))

    # ---------- mll mínima (se calcula ahora, se APLICA en la etapa 12) ----------
    # Para cada evento: la menor masa de entre TODOS los pares de carga opuesta
    # (6 pares posibles entre 4 leptones, de cualquier sabor).
    mll_min = np.full(n, np.inf)
    for a in range(4):
        for b in range(a + 1, 4):
            os_ = (q[a] + q[b]) == 0                 # ¿par de carga opuesta?
            mab = inv_mass(v[a], v[b])
            mll_min = np.where(os_, np.minimum(mll_min, mab), mll_min)

    # ---------- Etapa 9: emparejamiento en dos Z ----------
    # Los 4 leptones se pueden agrupar en 2 pares de 3 formas distintas:
    combos = [((0, 1), (2, 3)), ((0, 2), (1, 3)), ((0, 3), (1, 2))]

    def par_valido(a, b):
        """Par válido = mismo sabor (e-e o mu-mu) y carga opuesta."""
        return (pid[a] == pid[b]) & (q[a] + q[b] == 0)

    # Entre las combinaciones en que AMBOS pares son válidos, se elige la que
    # tenga el par más cercano a la masa del Z (ese par es Z1 y el otro Z2).
    mejor_score = np.full(n, np.inf)   # |mZ1 - MZ| de la mejor combinación hasta ahora
    mZ1 = np.full(n, np.nan)
    mZ2 = np.full(n, np.nan)
    for (a, b), (c, d) in combos:
        valido_c = par_valido(a, b) & par_valido(c, d)
        mab = inv_mass(v[a], v[b])
        mcd = inv_mass(v[c], v[d])
        ab_es_z1 = np.abs(mab - MZ) < np.abs(mcd - MZ)   # ¿(a,b) está más cerca de MZ?
        z1 = np.where(ab_es_z1, mab, mcd)
        z2 = np.where(ab_es_z1, mcd, mab)
        score = np.abs(z1 - MZ)
        mejor = valido_c & (score < mejor_score)
        mejor_score = np.where(mejor, score, mejor_score)
        mZ1 = np.where(mejor, z1, mZ1)
        mZ2 = np.where(mejor, z2, mZ2)

    # Si ninguna combinación fue válida, mZ1 queda NaN y el evento se descarta.
    cerrar(ETAPAS[9], np.isfinite(mZ1))
    # ---------- Etapas 10 y 11: ventanas de masa de Z1 y Z2 ----------
    cerrar(ETAPAS[10], (mZ1 > C["mz1_min"]) & (mZ1 < C["mz1_max"]))
    cerrar(ETAPAS[11], (mZ2 > C["mz2_min"]) & (mZ2 < C["mz2_max"]))
    # ---------- Etapa 12: mll mínima ----------
    cerrar(ETAPAS[12], mll_min > C["mll_min"])
    # ---------- Etapa 13: m4l mínima ----------
    cerrar(ETAPAS[13], m4l > C["m4l_min"])
    return res


def procesar(rutas, nombre, con_claves):
    """
    Lee uno o varios CSV por bloques y devuelve {etapa: (n, m4l)}.

    - Va acumulando, por etapa, el conteo y los valores de m4l de cada bloque.
    - Con con_claves=True (datos reales) se quitan los eventos duplicados entre
      archivos en CADA etapa, usando la clave (run, luminosityBlock, event);
      de cada evento repetido se conserva la primera aparición.
    """
    acum = {e: {"n": 0, "m4l": [], "claves": []} for e in ETAPAS}
    leidos = 0
    for ruta in rutas:
        for bloque in pd.read_csv(ruta, chunksize=CHUNKSIZE,
                                  usecols=lambda c: c in NECESARIAS):
            res = evaluar(bloque, CUTS, con_claves)
            for e, (m, cl, n) in res.items():
                a = acum[e]
                a["n"] += n
                if m is not None:
                    a["m4l"].append(m)
                if cl is not None:
                    a["claves"].append(cl)
            leidos += len(bloque)
            print(f"  {nombre}: {leidos} filas leidas", end="\r")
    print()

    salida = {}
    for e in ETAPAS:
        a = acum[e]
        m4l = np.concatenate(a["m4l"]) if a["m4l"] else np.array([], np.float32)
        if con_claves:
            # Eliminación de duplicados: np.unique devuelve el índice de la primera
            # aparición de cada clave; con esos índices se filtran m4l y el conteo.
            cl = np.concatenate(a["claves"]) if a["claves"] else np.array([], np.uint64)
            _, idx = np.unique(cl, return_index=True)
            n = len(idx)
            if m4l.size:
                m4l = m4l[idx]
        else:
            n = a["n"]
        salida[e] = (n, m4l)
    return salida


def peso_por_evento(nombre):
    """Peso de un evento simulado: w = L * sigma * k / N_generados."""
    m = MUESTRAS[nombre]
    return LUMI * m["xsec"] * m["k"] / m["nevt"]


# =============================================================================
# TABLAS
# =============================================================================

def tablas_cutflow(res_sim, res_datos):
    """
    Construye las dos tablas cutflow (filas = cortes, columnas = muestras):
      - tabla cruda:     eventos que sobreviven (MC SIN ponderar; datos tal cual).
      - tabla ponderada: eventos ESPERADOS a 11.58 fb^-1 (MC multiplicado por su
                         peso; los datos reales no se ponderan).
    Columnas: Higgs, los 3 canales ZZ y Datos reales.
    """
    crudo, pond = {}, {}
    for nombre, res in res_sim.items():
        col = NOMBRES_COLUMNA[nombre]
        w = peso_por_evento(nombre)
        crudo[col] = {e: res[e][0] for e in ETAPAS}
        pond[col] = {e: res[e][0] * w for e in ETAPAS}
    crudo["Datos reales"] = {e: res_datos[e][0] for e in ETAPAS}
    pond["Datos reales"] = {e: res_datos[e][0] for e in ETAPAS}
    t_crudo = pd.DataFrame(crudo)
    t_pond = pd.DataFrame(pond).round(3)
    t_crudo.index.name = "Filtro"
    t_pond.index.name = "Filtro"
    return t_crudo, t_pond


# =============================================================================
# BINNING (bordes de los bins del histograma)
# =============================================================================

def construir_bins(res_sim, res_datos):
    """
    Define UN solo binning para todas las gráficas (así el eje X es igual en todas).
    - Con VENTANA: 36 bins de igual ancho entre sus dos extremos.
    - Con VENTANA = None: bins de ANCHO_BIN GeV desde 0 hasta XMAX (o hasta el
      m4l más grande encontrado).
    """
    if VENTANA is not None:
        return np.linspace(VENTANA[0], VENTANA[1], 37)   # 37 bordes = 36 bins
    xmax = XMAX
    if xmax is None:
        # la etapa 1 contiene a todos los eventos válidos, así que ahí está el m4l máximo
        maximos = [r[ETAPAS[1]][1].max()
                   for r in list(res_sim.values()) + [res_datos] if r[ETAPAS[1]][1].size]
        xmax = max(maximos) if maximos else 200.0
    tope = np.ceil(xmax / ANCHO_BIN) * ANCHO_BIN + ANCHO_BIN
    return np.arange(0, tope, ANCHO_BIN)


# =============================================================================
# GRÁFICAS
# =============================================================================

def hists_mc(res_sim, etapa):
    """
    Histogramas del MC para una etapa, ponderados por luminosidad y sección eficaz.
    Devuelve (h_zz, h_zz + h_higgs), bin a bin:
      h_zz  : suma de los tres canales ZZ (el fondo).
      h_tot : fondo + señal Higgs apilados.
    """
    h_zz = np.zeros(len(BINS) - 1)
    h_h = np.zeros(len(BINS) - 1)
    for nombre, res in res_sim.items():
        m4l = res[etapa][1]
        h, _ = np.histogram(m4l, bins=BINS, weights=np.full(len(m4l), peso_por_evento(nombre)))
        if MUESTRAS[nombre]["tipo"] == "senal":
            h_h += h
        else:
            h_zz += h
    return h_zz, h_zz + h_h


def dibujar_datos(ax, res_datos, etapa, grande=True):
    """
    Histograma de barras de los datos reales unidos para una etapa.
    El eje Y se ajusta solo. 'grande=True' añade etiquetas y leyenda
    (figura individual); 'grande=False' es para los paneles pequeños.
    """
    m4l = res_datos[etapa][1]
    n_visibles = ((m4l >= BINS[0]) & (m4l <= BINS[-1])).sum()   # eventos dentro del eje X
    borde = 0.8 if len(BINS) < 100 else 0.2   # con muchos bins, borde más fino
    ax.hist(m4l, bins=BINS, color="steelblue", edgecolor="black", linewidth=borde,
            label=f"Datos ({n_visibles} eventos)")
    ax.set_xlim(BINS[0], BINS[-1])
    if grande:
        ax.set_xlabel("Mass 4 leptons / GeV", fontsize=12)
        ax.set_ylabel("N_Events", fontsize=12)
        ax.legend()


def dibujar_mc(ax, res_sim, etapa, grande=True):
    """
    Gráfica del MC al estilo CMS Open Data:
      - ZZ (los 3 canales juntos): histograma azul relleno.
      - ZZ + Higgs (mH = 125 GeV): línea roja.
    El eje Y se ajusta a la altura de esta etapa (+25 % de margen).
    """
    h_zz, h_tot = hists_mc(res_sim, etapa)
    ax.stairs(h_tot, BINS, color="red", linewidth=2, label="m$_H$ = 125 GeV")
    ax.stairs(h_zz, BINS, fill=True, facecolor="#9ecae1", edgecolor="black",
              linewidth=1.5 if len(BINS) < 100 else 0.5, label="ZZ")
    ax.set_xlim(BINS[0], BINS[-1])
    ax.set_ylim(0, (h_tot.max() if h_tot.max() > 0 else 1) * 1.25)
    if grande:
        ax.set_xlabel("Mass 4 leptons / GeV", fontsize=12)
        ax.set_ylabel("N_Events", fontsize=12)
        ax.text(0.02, 1.02, "CMS Open Data", transform=ax.transAxes, fontsize=12, fontweight="bold")
        ax.text(0.98, 1.02, r"$\sqrt{s}$ = 8 TeV, L$_{int}$ = 11.6 fb$^{-1}$",
                transform=ax.transAxes, fontsize=11, ha="right")
        ax.legend(frameon=False, loc="upper right", bbox_to_anchor=(0.97, 0.95))


def figura_etapa(dibujar, res, etapa, k, titulo, archivo):
    """Guarda en PNG la gráfica de UNA etapa (k = número de etapa) usando la función 'dibujar'."""
    fig, ax = plt.subplots(figsize=(8, 6))
    dibujar(ax, res, etapa, grande=True)
    fig.suptitle(f"{titulo} - {k}. {etapa} ({ETIQUETA})", fontsize=12)
    fig.tight_layout(rect=[0, 0, 1, 0.95])
    fig.savefig(archivo, dpi=150)
    if MOSTRAR:
        plt.show()
    plt.close(fig)


def panel(dibujar, res, titulo, archivo):
    """Todas las etapas en una sola figura (cada panel con su propia escala Y)."""
    etapas = ETAPAS[1:]     # la etapa 0 no tiene m4l, no se grafica
    ncols = 4
    nrows = int(np.ceil(len(etapas) / ncols))
    fig, axs = plt.subplots(nrows, ncols, figsize=(4.2 * ncols, 3.4 * nrows))
    for ax, (k, e) in zip(axs.ravel(), enumerate(etapas, 1)):
        dibujar(ax, res, e, grande=False)
        ax.set_title(f"{k}. {e}", fontsize=8)
    for ax in axs.ravel()[len(etapas):]:
        ax.axis("off")      # apaga los ejes sobrantes de la cuadrícula
    fig.suptitle(f"{titulo}: m4l tras cada filtro ({ETIQUETA})")
    fig.tight_layout()
    fig.savefig(archivo, dpi=130)
    if MOSTRAR:
        plt.show()
    plt.close(fig)


# =============================================================================
# PROGRAMA PRINCIPAL
# =============================================================================

if __name__ == "__main__":
    os.makedirs(SALIDA, exist_ok=True)

    # 1) Simulaciones: cada muestra se procesa por separado (sin quitar duplicados)
    res_sim = {}
    for nombre, ruta in RUTAS.items():
        print(f"Procesando {nombre}...")
        res_sim[nombre] = procesar([ruta], nombre, con_claves=False)

    # 2) Datos reales: los 4 CSV se procesan juntos, quitando eventos repetidos
    print("Procesando datos reales (4 datasets unidos)...")
    res_datos = procesar(DATOS_REALES, "Datos reales", con_claves=True)

    # 3) Tablas cutflow (se imprimen y se guardan como CSV)
    rango = "sin corte en m4l" if VENTANA is None else f"m4l en {VENTANA}"
    t_crudo, t_pond = tablas_cutflow(res_sim, res_datos)
    print(f"\n=== CUTFLOW, eventos crudos, {rango} ({ETIQUETA}) ===")
    print(t_crudo.to_string())
    print(f"\n=== CUTFLOW, eventos esperados a 11.58 fb^-1, {rango} ({ETIQUETA}) ===")
    print(t_pond.to_string())
    t_crudo.to_csv(SALIDA + f"cutflow_crudo_{ETIQUETA}.csv")
    t_pond.to_csv(SALIDA + f"cutflow_ponderado_{ETIQUETA}.csv")
    # t_crudo.to_latex(SALIDA + f"cutflow_crudo_{ETIQUETA}.tex")   # descomenta si la quieres en LaTeX

    # 4) Binning común (eje X igual en todas las gráficas)
    BINS = construir_bins(res_sim, res_datos)
    print(f"\nEje X: {BINS[0]:.0f} a {BINS[-1]:.0f} GeV ({len(BINS) - 1} bins)")

    # 5) Un histograma por filtro (datos y MC), cada uno con su propia escala Y
    for k, etapa in enumerate(ETAPAS[1:], 1):
        figura_etapa(dibujar_datos, res_datos, etapa, k, "Datos reales",
                     SALIDA + f"datos_etapa{k:02d}_{ETIQUETA}.png")
        figura_etapa(dibujar_mc, res_sim, etapa, k, "Simulaciones",
                     SALIDA + f"mc_etapa{k:02d}_{ETIQUETA}.png")

    # 6) Paneles con todas las etapas juntas
    panel(dibujar_datos, res_datos, "Datos reales", SALIDA + f"datos_panel_{ETIQUETA}.png")
    panel(dibujar_mc, res_sim, "Simulaciones (ZZ + Higgs)", SALIDA + f"mc_panel_{ETIQUETA}.png")